# Example steps and codes running hypoddpy for earthquake relocation

In [1]:
#import needed packages.
""" 
    Download hypoDD at https://www.ldeo.columbia.edu/~felixw/hypoDD.html
    Or use the copy included in this python interface package.
"""
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from hypoxpy.utils import tic, toc
import hypoxpy.HypoDDCore as hypoddcore
import matplotlib.pyplot as plt

In [2]:
binpath='/Users/xtyang/bin' #path to hypoDD binaries.
indir='input'
outdir='output'

namebase='EQ_HYP'
station_file=os.path.join(indir,'EQ_HYP_station_original.csv')
station_file_reformat=os.path.join(indir,'station.dat')
phase_file=os.path.join(indir,'EQ_HYP_phase_original.dat') #input original phase file.
phase_file_reformat=f'{indir}/{namebase}_phase_ready.pha' #output reformatted phase file for hypoDD.
phase_file_updated=f'{outdir}/{namebase}_phase_relocated.pha' #output updated phase file after relocation.
dep_corr = 5

#
cleanup=True #whether to clean up intermediate files.

#subset params
time_range='20190704-20190710'
lat_range = [35.45,36.05]
lon_range = [-117.8,-117.25]


In [3]:
print("[TIMER] HypoDD started")

# -------------------------------
# 2. Configure hypoDD running settings
# -------------------------------
hypoddconfig = hypoddcore.HypoDDConfig(binpath=binpath,indir=indir,outdir=outdir,namebase=namebase,
    station_file=station_file,phase_file=phase_file_reformat)

# 3. reformat station and phase files
hypoddcore.reformat_stationfile(station_file, station_file_reformat)

hypoddcore.reformat_phasefile(hypoddconfig, phase_file, phase_file_out=phase_file_reformat, \
                              time_range=time_range, lat_range=lat_range, lon_range=lon_range)

# -------------------------------
# 4. hypoDD (core relocation)
# -------------------------------
t0 = tic()
print(f"Running hypoDD ")
out_catalog = hypoddconfig.run(cleanup=cleanup)
toc(t0, f"Run hypoDD")

# 5. update original phase file with hypoDD relocation results
hypoddcore.update_phasefile_with_reloc(phase_file, out_catalog, phase_file_updated)




[TIMER] HypoDD started


IndexError: list index out of range

In [ ]:
quakes=pd.read_csv(out_catalog,header=None,names=["datetime","latitude","longitude","depth","magnitude","evid"])
print(quakes)

In [ ]:
plt.scatter(quakes.longitude,quakes.latitude,5*np.power(2,quakes.magnitude),quakes.depth,edgecolors='k')
plt.colorbar(label='depth (km)')
plt.show()